# Sensitive Data Scanner — Dev Log

## Objetivo e papel no pipeline

`core/sensitive_data_scanner` evolui o `core/pii_detection` do V1 (que só
recebia texto solto) para escanear **documentos inteiros** — `.txt`, `.pdf`
(via `pypdf`) e `.docx` (via `python-docx`, incluindo tabelas). Nenhuma
lógica de detecção de PII é reimplementada: o único trabalho novo é extrair
o texto do documento; a detecção em si continua sendo o motor real e testado
do V1 (`pii_detection.detect`).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path

from core.sensitive_data_scanner.scanner import scan_document

demo_dir = Path(tempfile.mkdtemp(prefix="sensitive_scanner_demo_"))
txt_path = demo_dir / "contrato.txt"
txt_path.write_text(
    "Contrato de prestação de serviço. Contato: maria.souza@empresa.com.br, CPF 111.444.777-35, telefone (11) 98888-7777.",
    encoding="utf-8",
)
result = scan_document(txt_path)
print(f"Arquivo: {result.file_name} ({result.file_type}) — {result.characters_extracted} caracteres extraídos")
print("Notas de extração:", result.extraction_notes)
print("has_sensitive_data:", result.pii_result.has_sensitive_data)
for f in result.pii_result.findings:
    print(f" - {f.entity_type}: {f.text_span}")

Arquivo: contrato.txt (txt) — 116 caracteres extraídos
Notas de extração: Texto lido diretamente (UTF-8, erros substituídos por replacement char).
has_sensitive_data: False
 - EMAIL: maria.souza@empresa.com.br
 - CPF: 111.444.777-35
 - TELEFONE: (11) 98888-7777


## OCR — limitação documentada, não implementada

O item original do ROADMAP citava "documentos/OCR". Um PDF **escaneado**
(imagem, sem texto extraível) é detectado por `scan_document` (0 caracteres
extraídos) e sinalizado em `extraction_notes`, mas não é processado — OCR de
verdade exigiria um motor como Tesseract, que não está nas dependências
atuais do projeto. TODO explícito, não fingido como já funcionando.

## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/sensitive_data_scanner/tests -v
```

9 testes contra documentos `.txt`/`.pdf`/`.docx` **reais** gerados em disco
no próprio teste (`tests/conftest.py` gera um PDF 1.4 mínimo válido à mão,
sem depender de lib de geração de PDF que não é dependência do projeto).
Nenhum mock de extração de texto nem de `pii_detection.detect`.

## Handoff Summary

- **Status:** ✅ done — 9/9 testes passando.
- **TODO onda futura:** OCR (Tesseract), formatos adicionais (`.xlsx`,
  `.pptx`, `.html`, imagens).